In [ ]:
from omegaconf import OmegaConf
from jupyterscad import view
from itertools import accumulate

from solid2 import square, cube

from configuration import schema, ConfigSchema

In [ ]:
yaml_config = OmegaConf.load("default.conf.yaml")
conf = OmegaConf.merge(schema, yaml_config)
conf

In [ ]:
def generate_keys_row(
    plate_width: float,
    plate_length: float,
    key_sep_distances: list[float],
    y_offset: float,
    conf: ConfigSchema,
):
    u = conf.mount_u
    mx_hole = square([u, u]).translateY(y_offset)
    mounting_plate = square([plate_width, plate_length])

    for sep in accumulate(key_sep_distances):
        mounting_plate -= mx_hole.translateX(sep)

    return mounting_plate.linear_extrude(conf.mount_plate_width)

In [ ]:
wk_total_width = conf.white_key_dims.width * conf.dist_u
single_key_len = conf.white_key_dims.length
white_plate_width = wk_total_width * 7
white_plate_len = single_key_len * conf.dist_u
distances = [(wk_total_width - conf.mount_u) / 2] + [wk_total_width] * 6
white_mount_plate = generate_keys_row(
    white_plate_width,
    white_plate_len,
    distances,
    (white_plate_len - conf.mount_u) / 2,
    conf,
) + cube([white_plate_width, conf.mount_plate_width, conf.base_height_mm]).down(
    conf.base_height_mm
)

view(white_mount_plate)

In [ ]:
bk_total_width = conf.black_key_dims.width * conf.dist_u
bk_len = conf.black_key_dims.length * conf.dist_u
bw_diff = conf.white_black_keys_offset_mm + conf.mount_plate_width
distances = [
    wk_total_width - conf.mount_u / 2,
    wk_total_width,
    wk_total_width * 2,
    wk_total_width,
    wk_total_width,
]
black_mount_plate = (
    generate_keys_row(
        white_plate_width,
        conf.dist_u,
        distances,
        (conf.dist_u - conf.mount_u) / 2,
        conf,
    )
    + cube([white_plate_width, conf.mount_plate_width, bw_diff]).down(bw_diff)
    + cube([white_plate_width, conf.mount_plate_width, bw_diff + conf.base_height_mm])
    .down(bw_diff + conf.base_height_mm)
    .translateY(conf.dist_u - conf.mount_plate_width)
)

view(
    black_mount_plate
    + white_mount_plate.translate(
        [
            0,
            -white_plate_len,
            -conf.white_black_keys_offset_mm - conf.mount_plate_width,
        ]
    )
)

In [ ]:
# TODO: each octave is indivisible part
#       but it is possible to generate half of the octave
#       currently I have 35 switches, so the max I can get is 2.5 octaves.
#       30~ keys + 5 mods (octave up, octave down, maybe some play, record, etc)

